In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
%cd /content/drive/MyDrive/ColabNotebooks/IS110_BPerformace_Borines/

/content/drive/MyDrive/ColabNotebooks/IS110_BPerformace_Borines


In [5]:
!pwd

/content/drive/MyDrive/ColabNotebooks/IS110_BPerformace_Borines


In [6]:
!mkdir EmsDashboard

In [7]:
%cd EmsDashboard/

/content/drive/MyDrive/ColabNotebooks/IS110_BPerformace_Borines/EmsDashboard


In [8]:
!pip install virtualenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.0/469.0 kB 29.9 MB/s eta 0:00:00


In [9]:
!python -m virtualenv venv

created virtual environment CPython3.12.13.final.0-64-x86_64 in 13175ms
  creator CPython3Posix(dest=/content/drive/MyDrive/ColabNotebooks/IS110_BPerformace_Borines/EmsDashboard/venv, clear=False, no_vcs_ignore=False, global=False)
  seeder FromAppData(download=False, pip=bundle, via=copy, app_data_dir=/root/.cache/virtualenv)
    added seed packages: pip==26.1.1
  activators BashActivator,CShellActivator,FishActivator,NushellActivator,PowerShellActivator,PythonActivator,XonshActivator


In [10]:
!ls

venv


In [11]:
!source venv/bin/activate

In [12]:
%%writefile requirements.txt
pandas
numpy
plotly
dash==3.2.0
dash-bootstrap-components
openpyxl

Writing requirements.txt


In [13]:
!cat requirements.txt

pandas
numpy
plotly
dash==3.2.0
dash-bootstrap-components
openpyxl


In [14]:
!pip freeze > requirements.txt

In [15]:
!python --version

Python 3.12.13


In [16]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, HTML
print('Done')

Done


In [17]:
np.random.seed(42)
random.seed(42)

In [18]:
rooms_config = {
    'Standard Room': ['Room 01', 'Room 02', 'Room 03', 'Room 04', 'Room 05', 'Room 06', 'Room 07', 'Room 08'],
    'Deluxe Room':   ['Room 09', 'Room 10', 'Room 11', 'Room 12', 'Room 13', 'Room 14'],
    'Suite Room':    ['Room 15', 'Room 16']
}

base_prices = {
    'Room 01': 3500, 'Room 02': 3000, 'Room 03': 3500, 'Room 04': 3000,
    'Room 05': 3500, 'Room 06': 3500, 'Room 07': 3500, 'Room 08': 3500,
    'Room 09': 4500, 'Room 10': 4500, 'Room 11': 4000, 'Room 12': 4500,
    'Room 13': 4500, 'Room 14': 4500, 'Room 15': 5500, 'Room 16': 5500
}

payment_statuses = ['Paid', 'Unpaid', 'Overdue']
visitors_pool    = ['Mark Vega', 'Anna Ramos', 'John Doe', 'Maria Santos', 'None']
tenants_pool     = [f'Tenant_{i:03d}' for i in range(1, 81)]


rows = []
start = datetime(2026, 1, 1)

for i in range(1, 1001):
    category = random.choice(list(rooms_config))
    room = random.choice(rooms_config[category])
    status = random.choice(payment_statuses)
    price = base_prices[room]
    date = start + timedelta(days=random.randint(0, 150))

    rows.append({
        'transaction_id': f'EMS-{1000+i}',
        'date': date.strftime('%Y-%m-%d'),
        'tenant_name': random.choice(tenants_pool),
        'room': room,
        'room_category': category,
        'payment_status': status,
        'amount': price,
        'visitor_name': random.choice(visitors_pool),
        'duration_hours': random.randint(1, 5) if random.choice([True, False]) else 0
    })

df = pd.DataFrame(rows)
print(f"Generated {len(df)} initial dictionary elements.")

Generated 1000 initial dictionary elements.


In [19]:
import os
os.makedirs('data', exist_ok=True)
df.to_csv('data/ems_boarding_data.csv', index=False)
print(f'Generated {len(df)} rows -> data/ems_boarding_data.csv')

Generated 1000 rows -> data/ems_boarding_data.csv


In [20]:
import pandas as pd

df = pd.read_csv('data/ems_boarding_data.csv')

df['date'] = pd.to_datetime(df['date'])
df['year']        = df['date'].dt.year
df['month']       = df['date'].dt.month
df['month_name']  = df['date'].dt.strftime('%b %Y')
df['month_order'] = df['date'].dt.to_period('M')
df['amount'] = df['amount'].fillna(0)
df.dropna(subset=['transaction_id', 'date'], inplace=True)
df['duration_hours'] = df['duration_hours'].astype(int)
df['amount']         = df['amount'].astype(float)


print(f'Loaded {len(df)} rows, date range: {df.date.min().date()} to {df.date.max().date()}')

Loaded 1000 rows, date range: 2026-01-01 to 2026-05-31


In [21]:
monthly = (
    df[df['payment_status'] == 'Paid']
    .groupby(['month_order', 'month_name'], as_index=False)
    .agg(revenue=('amount', 'sum'), transactions=('transaction_id', 'count'))
    .sort_values('month_order')
)

by_category = (
    df.groupby('room_category', as_index=False)
    .agg(revenue=('amount', 'sum'))
    .sort_values('revenue', ascending=False)
)
print('Data structures aggregation complete.')

Data structures aggregation complete.


In [22]:
def make_kpi_summary():
    kpi_html = """
    <div style="display: flex; gap: 10px; justify-content: space-between; font-family: Arial, sans-serif; margin-top: 10px;">
        <div style="border: 1px solid #cbd5e1; padding: 12px; flex: 1; text-align: center; border-radius: 6px; background:#fff;">
            <span style="font-size:12px; color:#64748b; font-weight:bold;">TOTAL TENANTS</span><br>
            <span style="font-size:20px; font-weight:bold; color:#0f172a;">👥 28 Active</span>
        </div>
        <div style="border: 1px solid #cbd5e1; padding: 12px; flex: 1; text-align: center; border-radius: 6px; background:#fff;">
            <span style="font-size:12px; color:#64748b; font-weight:bold;">OCCUPANCY RATE</span><br>
            <span style="font-size:20px; font-weight:bold; color:#0f172a;">🛏️ 87.5%</span><br>
            <small style="color: #64748b;">(14 / 16 Rooms)</small>
        </div>
        <div style="border: 1px solid #cbd5e1; padding: 12px; flex: 1; text-align: center; border-radius: 6px; background:#fff;">
            <span style="font-size:12px; color:#64748b; font-weight:bold;">MONTHLY REVENUE</span><br>
            <span style="font-size:20px; font-weight:bold; color:#0f172a;">₱42,000.00</span>
        </div>
        <div style="border: 1px solid #f87171; padding: 12px; flex: 1; text-align: center; border-radius: 6px; background:#fff;">
            <span style="font-size:12px; color:#ef4444; font-weight:bold;">OUTSTANDING BALANCES</span><br>
            <span style="font-size:20px; font-weight:bold; color:#ef4444;">₱4,500.00</span>
        </div>
    </div>
    """
    display(HTML(kpi_html))

make_kpi_summary()

In [27]:
def make_occupancy_gauge(dataframe):
    """Gauge chart: real-time space occupancy rate."""
    fig = go.Figure(go.Indicator(
        mode = "gauge+number",
        value = 87.5,
        title = {'text': "EMS Occupancy Capacity Rate (%)", 'font': {'size': 16}},
        gauge = {
            'axis': {'range': [0, 100], 'ticksuffix': "%"},
            'bar': {'color': "#2E75B6"},
            'steps': [
                {'range': [0, 70], 'color': "#cbd5e1"},
                {'range': [70, 100], 'color': "#93c5fd"}
            ]
        }
    ))
    fig.update_layout(
        plot_bgcolor='white',
        paper_bgcolor='white',
        height=300,
        margin=dict(l=30, r=30, t=50, b=30)
    )
    return fig

In [64]:
fig_occupancy_gauge = make_occupancy_gauge(df)
fig_occupancy_gauge.show()

In [29]:
def make_revenue_trend(dataframe):
    """Line chart: monthly total revenue with markers."""
    monthly_rev = (
        dataframe[dataframe['payment_status'] == 'Paid']
        .groupby(['month_order', 'month_name'], as_index=False)
        .agg(revenue=('amount', 'sum'))
        .sort_values('month_order')
    )
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=monthly_rev['month_name'],
        y=monthly_rev['revenue'],
        mode='lines+markers',
        name='Monthly Revenue',
        line=dict(color='#2E75B6', width=3),
        marker=dict(size=8, color='#1F4E79'),
        fill='tozeroy',
        fillcolor='rgba(46,117,182,0.1)'
    ))
    fig.update_layout(
        title='Monthly Revenue Trend (EMS Boarding House)',
        xaxis_title='Month',
        yaxis_title='Revenue (PHP)',
        plot_bgcolor='white',
        paper_bgcolor='white',
        hovermode='x unified',
        yaxis=dict(tickformat=',.0f'),
        height=500
    )
    return fig

In [50]:
fig_revenue_trend = make_revenue_trend(df)
fig_revenue_trend.show()

In [31]:
def make_payment_pie(dataframe):
    status_summary = dataframe.groupby('payment_status', as_index=False).agg(total=('amount', 'sum'))
    fig = px.pie(
        status_summary,
        names='payment_status',
        values='total',
        title='Collection Status Financial Performance Share',
        color='payment_status',
        color_discrete_map={'Paid': '#10b981', 'Unpaid': '#ef4444', 'Overdue': '#f59e0b'}
    )
    fig.update_layout(plot_bgcolor='white', height=500)
    return fig

In [32]:
fig_payment_pie = make_payment_pie(df)
fig_payment_pie.show()

In [33]:
def make_tenant_bar(dataframe):

    by_cat = dataframe.groupby('room_category', as_index=False).agg(revenue=('amount', 'sum')).sort_values('revenue')
    fig = px.bar(
        by_cat,
        x='revenue',
        y='room_category',
        orientation='h',
        color='revenue',
        color_continuous_scale='Blues',
        title='Tenant Rental Value Distribution by Room Category',
        labels={'revenue': 'Total Yield (PHP)', 'room_category': ''},
        text_auto=',.0f'
    )
    fig.update_layout(coloraxis_showscale=False, plot_bgcolor='white', height=500)
    return fig

In [34]:
fig_tenant_bar = make_tenant_bar(df)
fig_tenant_bar.show()

In [35]:
def make_room_donut(dataframe):
    room_snap = dataframe.drop_duplicates(subset=['room']).groupby('room_category', as_index=False).agg(count=('room', 'count'))
    fig = px.pie(
        room_snap,
        names='room_category',
        values='count',
        title='Room Inventory Allocation Summary Share',
        hole=0.4,
        color_discrete_sequence=px.colors.sequential.Blues_r
    )
    fig.update_traces(textposition='outside', textinfo='percent+label')
    fig.update_layout(height=500)
    return fig

In [51]:
fig_room_donut = make_room_donut(df)
fig_room_donut.show()

In [40]:
def make_visitor_table(dataframe):
    real_visitors = dataframe[dataframe['visitor_name'] != 'None'][['date', 'tenant_name', 'room', 'visitor_name', 'duration_hours']].head(5)
    real_visitors.columns = ['Date Logged', 'Host Tenant', 'Room Assigned', 'Guest Name', 'Stay Duration (Hrs)']
    return real_visitors

In [52]:
visitor_logs_table = make_visitor_table(df)
print("\nSECURITY TRACKING SYSTEM LIVE LOG RECORDS:")
display(visitor_logs_table)


SECURITY TRACKING SYSTEM LIVE LOG RECORDS:


,Date Logged,Host Tenant,Room Assigned,Guest Name,Stay Duration (Hrs)
0,2026-03-12,Tenant_032,Room 15,Anna Ramos,1
1,2026-04-19,Tenant_005,Room 15,Mark Vega,2
2,2026-02-20,Tenant_070,Room 01,Maria Santos,4
3,2026-02-10,Tenant_055,Room 16,John Doe,0
4,2026-01-27,Tenant_012,Room 04,Maria Santos,3


In [42]:
!pip install dash==3.2.0 dash_bootstrap_components

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.0/204.0 kB 21.0 MB/s eta 0:00:00


In [58]:
import dash
from dash import Dash, dcc, html, dash_table, Input, Output
from google.colab import output
import dash_bootstrap_components as dbc
print('Done')

Done


In [59]:
app = Dash(
    __name__,
    external_stylesheets=[
        dbc.themes.BOOTSTRAP,
        'https://cdn.jsdelivr.net/npm/bootstrap-icons/font/bootstrap-icons.css'
    ],
    title='EMS Boarding House Dashboard'
)
print('Done')

Done


In [60]:
import plotly.graph_objects as go
import plotly.express as px

def make_revenue_trend(d):
    m = (d[d['payment_status'] == 'Paid']
         .groupby(['month_order','month_name'], as_index=False)
         .agg(revenue=('amount','sum')).sort_values('month_order'))
    fig = go.Figure(go.Scatter(
        x=m['month_name'], y=m['revenue'],
        mode='lines+markers', fill='tozeroy',
        line=dict(color='#2E75B6', width=3),
        marker=dict(size=8), fillcolor='rgba(46,117,182,0.1)'
    ))
    fig.update_layout(title='Monthly Revenue Trend (Paid Rentals)',
                      xaxis_title='Month', yaxis_title='Revenue (PHP)',
                      plot_bgcolor='white', hovermode='x unified',
                      yaxis=dict(tickformat=',.0f'), height=300)
    return fig

def make_category_bar(d):
    b = (d.groupby('room_category', as_index=False)
         .agg(revenue=('amount','sum')).sort_values('revenue'))
    fig = px.bar(b, x='revenue', y='room_category', orientation='h',
                 color='revenue', color_continuous_scale='Blues',
                 title='Rental Yield by Room Category', text_auto=',.0f',
                 labels={'revenue':'Revenue (PHP)','room_category':''})
    fig.update_layout(coloraxis_showscale=False, plot_bgcolor='white', height=300)
    return fig

def make_payment_pie(d):
    r = d.groupby('payment_status', as_index=False).agg(revenue=('amount','sum'))
    return px.pie(r, names='payment_status', values='revenue', hole=0.4,
                  title='Collection Financial Status',
                  color='payment_status',
                  color_discrete_map={'Paid': '#10b981', 'Unpaid': '#ef4444', 'Overdue': '#f59e0b'})

def make_room_donut(d):
    room_snap = d.drop_duplicates(subset=['room']).groupby('room_category', as_index=False).agg(count=('room', 'count'))
    fig = px.pie(room_snap, names='room_category', values='count', hole=0.4,
                 title='Room Allocation Summary Share',
                 color_discrete_sequence=px.colors.sequential.Blues_r)
    fig.update_traces(textposition='outside', textinfo='percent+label')
    fig.update_layout(height=300)
    return fig

def make_occupancy_gauge(d):
    fig = go.Figure(go.Indicator(
        mode = "gauge+number",
        value = 87.5,
        title = {'text': "EMS Occupancy Capacity Rate (%)", 'font': {'size': 14}},
        gauge = {
            'axis': {'range': [0, 100], 'ticksuffix': "%"},
            'bar': {'color': "#2E75B6"},
            'steps': [{'range': [0, 70], 'color': "#cbd5e1"},
                      {'range': [70, 100], 'color': "#93c5fd"}]
        }
    ))
    fig.update_layout(plot_bgcolor='white', paper_bgcolor='white', height=280,
                      margin=dict(l=30, r=30, t=50, b=30))
    return fig

def kpi_card(title, value, icon, color):
    return dbc.Card([
        dbc.CardBody([
            html.Div([
                html.I(className=f'bi {icon} fs-2', style={'color': color}),
                html.Div([
                    html.P(title, className='text-muted mb-0', style={'fontSize': '0.85rem'}),
                    html.H4(value, className='mb-0 fw-bold', style={'color': color})
                ], className='ms-3')
            ], className='d-flex align-items-center')
        ])
    ], className='shadow-sm h-100')

print('Chart functions successfully registered!')

Chart functions successfully registered!


In [61]:
app = Dash(
    __name__,
    external_stylesheets=[
        dbc.themes.BOOTSTRAP,
        'https://cdn.jsdelivr.net/npm/bootstrap-icons/font/bootstrap-icons.css'
    ],
    title='EMS Boarding House Dashboard'
)
print('Done')

Done


In [62]:
app.layout = dbc.Container([


    dbc.Row([
        dbc.Col(
            html.H2('📱 EMS Boarding House Management Dashboard',
                    className='text-white fw-bold py-3 mb-0 text-center'),
            style={'background': '#1F4E79'}
        )
    ], className='mb-4'),


    dbc.Row([
        dbc.Col([
            html.Label('Filter Room Type:', className='fw-semibold'),
            dcc.Dropdown(
                id='category-filter',
                options=[{'label': c, 'value': c} for c in sorted(df['room_category'].unique())],
                multi=True,
                placeholder='All Room Layouts...'
            )
        ], md=4),
        dbc.Col([
            html.Label('Filter Payment Status:', className='fw-semibold'),
            dcc.Dropdown(
                id='status-filter',
                options=[{'label': s, 'value': s} for s in sorted(df['payment_status'].unique())],
                multi=True,
                placeholder='All Record Statuses...'
            )
        ], md=4),
        dbc.Col([
            html.Label('Date Range Picker:', className='fw-semibold'),
            dcc.DatePickerRange(
                id='date-filter',
                start_date=df['date'].min(),
                end_date=df['date'].max(),
                display_format='MMM DD, YYYY'
            )
        ], md=4)
    ], className='mb-4 p-3 bg-light rounded'),


    dbc.Row(id='kpi-row', className='mb-4'),


    dbc.Row([
        dbc.Col(dcc.Graph(id='occupancy-gauge'), md=4),
        dbc.Col(dcc.Graph(id='revenue-trend'), md=8)
    ], className='mb-4'),


    dbc.Row([
        dbc.Col(dcc.Graph(id='payment-pie'), md=4),
        dbc.Col(dcc.Graph(id='room-donut'), md=4),
        dbc.Col(dcc.Graph(id='category-bar'), md=4)
    ], className='mb-4'),


    dbc.Row([
        dbc.Col([
            html.H5('📋 Security Visitor Logs & Transaction Master Registry', className='fw-bold mb-3'),
            dash_table.DataTable(
                id='data-table',
                columns=[{'name': c.replace('_',' ').title(), 'id': c}
                         for c in ['transaction_id','date','tenant_name',
                                   'room','room_category','payment_status',
                                   'amount','visitor_name','duration_hours']],
                page_size=10,
                sort_action='native',
                filter_action='native',
                style_header={'backgroundColor': '#1F4E79',
                              'color': 'white', 'fontWeight': 'bold'},
                style_data_conditional=[
                    {'if': {'row_index': 'odd'}, 'backgroundColor': '#F0F4F8'}
                ],
                style_cell={'textAlign': 'left', 'padding': '8px',
                            'fontFamily': 'Arial', 'fontSize': '13px'}
            )
        ])
    ]),


    dbc.Row([
        dbc.Col(html.Div("🗺️ [🏠 Home]   [🏪 Sari-Sari Store]   [📱 GCash Ledger]   [⚙️ App Settings]",
                         className='text-center text-white py-3 font-monospace fw-bold',
                         style={'background':'#1F4E79','fontSize':'0.9rem'}), className='mt-4')
    ])

], fluid=True)
print('Layout loaded successfully')

Layout loaded successfully


In [63]:
@app.callback(
    Output('kpi-row',         'children'),
    Output('occupancy-gauge', 'figure'),
    Output('revenue-trend',   'figure'),
    Output('payment-pie',     'figure'),
    Output('room-donut',      'figure'),
    Output('category-bar',    'figure'),
    Output('data-table',      'data'),
    Input('category-filter',  'value'),
    Input('status-filter',    'value'),
    Input('date-filter',      'start_date'),
    Input('date-filter',      'end_date'),
)
def update_dashboard(categories, statuses, start_date, end_date):

    filtered = df.copy()

    if categories:
        filtered = filtered[filtered['room_category'].isin(categories)]

    if statuses:
        filtered = filtered[filtered['payment_status'].isin(statuses)]

    if start_date:
        filtered = filtered[filtered['date'] >= start_date]

    if end_date:
        filtered = filtered[filtered['date'] <= end_date]


    total_tenants = len(filtered['tenant_name'].unique())
    total_rev     = filtered[filtered['payment_status'] == 'Paid']['amount'].sum()
    unpaid_bal    = filtered[filtered['payment_status'] != 'Paid']['amount'].sum()

    kpis = dbc.Row([
        dbc.Col(kpi_card('Total Active Tenants', f'👥 {total_tenants} Accounts', 'bi-people', '#1F4E79'), md=3),
        dbc.Col(kpi_card('Occupancy Rate Status', '🛏️ 87.5% (14/16 Rooms)', 'bi-door-closed', '#2E75B6'), md=3),
        dbc.Col(kpi_card('Collected Revenue', f'₱{total_rev:,.2f}', 'bi-wallet2', '#28A745'), md=3),
        dbc.Col(kpi_card('Outstanding Balances', f'₱{unpaid_bal:,.2f}', 'bi-exclamation-triangle', '#DC3545'), md=3)
    ])


    gauge_fig = make_occupancy_gauge(filtered)
    trend_fig = make_revenue_trend(filtered)
    pie_fig   = make_payment_pie(filtered)
    donut_fig = make_room_donut(filtered)
    bar_fig   = make_category_bar(filtered)


    table_df = filtered[['transaction_id','date','tenant_name',
                         'room','room_category','payment_status',
                         'amount','visitor_name','duration_hours']].copy()
    table_df['date']   = table_df['date'].dt.strftime('%Y-%m-%d')
    table_df['amount'] = table_df['amount'].map('{:,.2f}'.format)

    return kpis, gauge_fig, trend_fig, pie_fig, donut_fig, bar_fig, table_df.to_dict('records')

if __name__ == '__main__':
    app.run(debug=True, jupyter_mode="inline")

<IPython.core.display.Javascript object>